# Lección 2.7 — Datos desbalanceados: cuando lo importante es raro

> **Módulo 2 · Los datos: el combustible** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 2.5 (división y estratificación) y 2.6 (fuga de datos)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar por qué la **exactitud** engaña cuando una clase es rara.
- Leer una tabla de aciertos y errores y calcular **detección** (*recall*) y **precisión** de forma intuitiva.
- Aplicar las estrategias principales: **pesos de clase**, **sobremuestreo**, **submuestreo** y **mover el umbral**.
- Evitar la fuga de datos al re-muestrear (solo se re-muestrea el entrenamiento).

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix
from sklearn.utils import resample

plt.rcParams["figure.figsize"] = (8, 4)

def reporte(y_real, y_pred, nombre=""):
    """Imprime exactitud, detección (recall) y precisión de la clase rara (1)."""
    vn, fp, fn, vp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    exactitud = (vp + vn) / len(y_real)
    deteccion = vp / (vp + fn) if vp + fn else 0
    precision = vp / (vp + fp) if vp + fp else 0
    print(f"{nombre:<22} exactitud {exactitud:6.1%} · detección {deteccion:6.1%} · precisión {precision:6.1%}"
          f"   (atrapados {vp}, escapados {fn}, falsas alarmas {fp})")

print("✅ Todo listo")

---
## 1. 📖 La intuición: el detector de fraudes perezoso

En un banco, **1 de cada 100** transacciones es un fraude. Un programador perezoso entrega este "detector":

```python
def detector(transaccion):
    return "no es fraude"
```

Acierta el **99 %** de las veces. ¡Exactitud excelente! Y, sin embargo, **no detecta ni un solo fraude**, que era lo único que importaba.

Esto es un problema de **datos desbalanceados**: una clase (la "normal") es mucho más frecuente que la otra (la "rara", que casi siempre es la importante). Pasa en muchísimos problemas reales:

| Problema | Clase rara | Proporción típica |
|---|---|---|
| Fraude con tarjeta | Transacción fraudulenta | < 1 % |
| Diagnóstico médico | Enfermo | 1–10 % |
| Abandono de clientes | Cliente que se va | 5–20 % |
| Fallas en máquinas | Falla | < 1 % |
| Spam | Correo spam | Variable |

### ✋ Ejercicio 1 — Calentamiento
Una prueba médica para una enfermedad que tiene **2 de cada 1,000** personas siempre dice "sano". ¿Qué exactitud tiene? ¿La usarías?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**99.8 %** de exactitud (acierta con los 998 sanos) y **0 %** de enfermos detectados. Es inútil: justo los 2 que necesitaban tratamiento se van a casa. **Con clases desbalanceadas, la exactitud por sí sola no sirve para juzgar un modelo.**
</details>

---
## 2. 📖 Mirar los errores por separado

En lugar de un solo número, hay que mirar **qué tipo** de aciertos y errores tiene el modelo. Pensemos en un detector de fraudes:

|  | El modelo dice **fraude** 🚨 | El modelo dice **normal** ✅ |
|---|---|---|
| **Era fraude** | ✔️ **Atrapado** | ❌ **Escapado** (el ladrón se fue con el dinero) |
| **Era normal** | ❌ **Falsa alarma** (cliente molesto) | ✔️ Correcto |

De ahí salen dos preguntas clave:
- **Detección** (*recall*): de todos los fraudes **reales**, ¿qué porcentaje atrapé? → atrapados / (atrapados + escapados)
- **Precisión** (*precision*): de todas mis **alarmas**, ¿qué porcentaje eran fraude de verdad? → atrapados / (atrapados + falsas alarmas)

> 📝 En la Lección 4.2 verás estas métricas con todo detalle (y sus nombres técnicos: verdaderos positivos, falsos negativos…). Por ahora, basta con la intuición.

### ✋ Ejercicio 2 — Calcula a mano
De 1,000 transacciones, 20 eran fraude. Un modelo lanzó **30 alarmas**: 15 eran fraude de verdad y 15 eran clientes normales.
1. ¿Cuántos fraudes se escaparon?
2. ¿Cuál es su detección?
3. ¿Cuál es su precisión?

✍️ *Tus cálculos:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. 20 − 15 = **5** escapados.
2. Detección = 15 / 20 = **75 %**: atrapó 3 de cada 4 fraudes.
3. Precisión = 15 / 30 = **50 %**: la mitad de sus alarmas fueron falsas.

Fíjate en que su exactitud sería (15 + 965) / 1,000 = 98 %… ¡**exactamente la misma** que el detector perezoso que dice siempre "normal" (980 / 1,000 = 98 %)! Y sin embargo es muchísimo más útil.
</details>

---
## 3. 📖 Qué pasa si entrenas sin pensar

Vamos a crear un conjunto de 5,000 transacciones con **5 %** de fraudes y entrenar una regresión logística normal.

### 🔮 Ejercicio 3 — ¿Qué tan bueno es?
Antes de ejecutar: ¿qué exactitud esperas del detector perezoso (`DummyClassifier`, que siempre dice "normal")? ¿Y la logística: tendrá una detección alta o baja?

✍️ *Mi apuesta:*

In [ ]:
X, y = make_classification(n_samples=5_000, n_features=8, n_informative=4, n_redundant=2,
                           weights=[0.95, 0.05], class_sep=2.0, flip_y=0.01, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print(f"Fraudes en entrenamiento: {y_tr.sum()} de {len(y_tr)} ({y_tr.mean():.1%})\n")

perezoso = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
logistica = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)

reporte(y_te, perezoso.predict(X_te), "Perezoso")
reporte(y_te, logistica.predict(X_te), "Logística normal")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Perezoso:** exactitud ≈**94.5 %**, detección **0 %**.
- **Logística normal:** exactitud ≈**96 %**… y detección de apenas ≈**38 %**: deja escapar a casi 2 de cada 3 fraudes (51 de 82).

¿Por qué? El modelo aprende a minimizar sus errores **totales**. Como el 95 % de los ejemplos son normales, la forma "barata" de equivocarse poco es decir "normal" casi siempre. Para el modelo, un fraude escapado cuesta lo mismo que una falsa alarma… pero para el banco no.
</details>

---
## 4. 📖 Estrategia 1: pesos de clase

La idea más simple: decirle al modelo que **equivocarse con la clase rara cuesta más**. Con `class_weight="balanced"`, scikit-learn le da a cada clase un peso inversamente proporcional a su frecuencia: aquí, cada fraude "pesa" unas **17 veces** más que una transacción normal (3,310 normales / 190 fraudes en el entrenamiento).

Es como si el modelo viera a cada fraude repetido 17 veces, pero sin copiar ningún dato.

### 🔮 Ejercicio 4 — El precio de detectar más
Antes de ejecutar: con pesos balanceados, ¿qué crees que pasará con la **detección**? ¿Y con la **precisión** y la **exactitud**?

✍️ *Mi apuesta:*

In [ ]:
balanceada = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")).fit(X_tr, y_tr)

reporte(y_te, logistica.predict(X_te), "Logística normal")
reporte(y_te, balanceada.predict(X_te), "Logística balanceada")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Detección:** sube muchísimo, de ≈38 % a ≈**85 %**.
- **Precisión:** baja, de ≈86 % a ≈**28 %**: ahora hay muchas falsas alarmas (de 5 a 178).
- **Exactitud:** baja de ≈96 % a ≈**87 %**.

**No hay almuerzo gratis** (Lección 1.6): detectar más fraudes cuesta más falsas alarmas. ¿Cuál versión es mejor? **Depende del negocio**: si un fraude escapado cuesta 10,000 y revisar una falsa alarma cuesta 50, la balanceada gana por mucho. Lo verás con números en los ejercicios finales.
</details>

---
## 5. 📖 Estrategia 2: re-muestrear (cambiar los datos de entrenamiento)

En vez de cambiar los pesos, se cambia **la cantidad** de ejemplos de cada clase:

| Estrategia | Qué hace | Ventaja | Riesgo |
|---|---|---|---|
| **Sobremuestreo** (*oversampling*) | Copia (al azar) ejemplos de la clase rara hasta equilibrar | No tira datos | Copias exactas → el modelo puede memorizarlas |
| **Submuestreo** (*undersampling*) | Tira ejemplos de la clase frecuente al azar | Entrena rápido | **Pierdes** mucha información |
| **SMOTE** | Crea ejemplos raros **sintéticos**, "a medio camino" entre dos ejemplos raros parecidos | Menos memorización que copiar | Puede inventar ejemplos poco realistas |

> ⚠️ **Regla de oro:** se re-muestrea **solo el entrenamiento**, **después** de dividir. El conjunto de prueba debe tener la proporción **real** (la que verás en la vida real). Lo comprobarás en la Sección 7.

In [ ]:
entrenamiento = pd.DataFrame(X_tr).assign(fraude=y_tr)
fraudes, normales = entrenamiento[entrenamiento.fraude == 1], entrenamiento[entrenamiento.fraude == 0]

# Sobremuestreo: copiamos fraudes al azar hasta igualar a los normales
sobre = pd.concat([normales, resample(fraudes, n_samples=len(normales), random_state=0)])
# Submuestreo: nos quedamos con tantos normales como fraudes hay
sub = pd.concat([resample(normales, n_samples=len(fraudes), replace=False, random_state=0), fraudes])

print(f"Original     : {len(normales):>5} normales · {len(fraudes):>5} fraudes")
print(f"Sobremuestreo: {(sobre.fraude == 0).sum():>5} normales · {(sobre.fraude == 1).sum():>5} fraudes")
print(f"Submuestreo  : {(sub.fraude == 0).sum():>5} normales · {(sub.fraude == 1).sum():>5} fraudes\n")

for nombre, datos in [("Sobremuestreo", sobre), ("Submuestreo", sub)]:
    m = make_pipeline(StandardScaler(), LogisticRegression()).fit(datos.drop(columns="fraude"), datos.fraude)
    reporte(y_te, m.predict(X_te), nombre)
reporte(y_te, balanceada.predict(X_te), "Pesos balanceados")

### ✋ Ejercicio 5
1. ¿Cuántos ejemplos de entrenamiento **normales** se tiraron en el submuestreo?
2. Compara los tres métodos. ¿Se parecen sus resultados?
3. Si los tres dan resultados parecidos, ¿cuál preferirías y por qué?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Se tiraron **3,120** normales: de 3,310 quedaron solo 190. ¡El 89 % del entrenamiento a la basura!
2. **Sobremuestreo** y **pesos** dan casi lo mismo (≈86 % de detección y ≈28–29 % de precisión): para un modelo como la logística, "pesar" un ejemplo 17 veces o copiarlo 17 veces es casi equivalente. El **submuestreo** queda algo peor (≈82 % y ≈21 %, con más falsas alarmas): aprendió con muchos menos ejemplos de transacciones normales.
3. Los **pesos de clase** suelen ser la primera opción: una sola línea, no cambian los datos y no hay riesgo de equivocarse al re-muestrear. El submuestreo es útil cuando tienes **millones** de ejemplos normales y entrenar con todos es muy lento.
</details>

### 📖 SMOTE (para Colab)
SMOTE vive en la librería `imbalanced-learn`, que no viene con scikit-learn. Si quieres probarlo en Google Colab:

```python
# !pip install imbalanced-learn        ← solo la primera vez
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline as make_pipeline_imb

modelo_smote = make_pipeline_imb(StandardScaler(), SMOTE(random_state=0), LogisticRegression())
modelo_smote.fit(X_tr, y_tr)          # SMOTE se aplica SOLO al entrenamiento, dentro del pipeline
reporte(y_te, modelo_smote.predict(X_te), "SMOTE")
```
Fíjate en que va **dentro de un pipeline**: así nunca toca los datos de prueba ni de validación (Lección 2.6).

---
## 6. 📖 Estrategia 3: mover el umbral

Muchos modelos no dicen solo "fraude / normal": dan una **probabilidad** ("esta transacción tiene 30 % de probabilidad de ser fraude"). Por defecto, se dispara la alarma si la probabilidad es **≥ 50 %**. Pero ese 50 % no es sagrado: si los fraudes son muy caros, conviene disparar con menos certeza.

Es como un detector de humo: lo puedes ajustar para que suene con poco humo (más alarmas, menos incendios escapados) o solo con mucho.

### 🔮 Ejercicio 6 — Bajar el umbral
Usaremos la **logística normal** (sin pesos) y solo cambiaremos el umbral. Antes de ejecutar: al bajar el umbral de 0.5 a 0.1, ¿qué pasa con la detección y con las falsas alarmas?

✍️ *Mi apuesta:*

In [ ]:
probabilidades = logistica.predict_proba(X_te)[:, 1]      # probabilidad de "fraude" para cada transacción

for umbral in [0.5, 0.3, 0.2, 0.1, 0.05]:                  # ✏️ prueba otros umbrales
    reporte(y_te, (probabilidades >= umbral).astype(int), f"umbral = {umbral}")

<details><summary><b>👀 Ver respuesta</b></summary>

Al bajar el umbral, la **detección sube** (de ≈38 % con 0.5 a ≈67 % con 0.1) y las **falsas alarmas también suben** (de 5 a 83). Con 0.05 ya se atrapan ≈78 % de los fraudes, pero con casi 300 falsas alarmas.

Moraleja: **mover el umbral** consigue un efecto parecido a los pesos de clase, sin reentrenar, y te deja elegir el punto exacto del equilibrio. ¿Cómo eliges? Con el **costo** de cada error (ejercicios finales y Lección 4.3).

> ⚠️ El umbral también es una decisión: elígelo con validación, **nunca** mirando el conjunto de prueba (Lección 2.5).
</details>

---
## 7. 📖 Qué pasa si… re-muestreas antes de dividir

Un error muy común: copiar los fraudes **antes** de dividir en entrenamiento y prueba. Las copias de un mismo fraude terminan en **los dos lados**, y el modelo las "reconoce" en la prueba (¡es una fuga por duplicados, Lección 2.6!).

### 🔮 Ejercicio 7 — ¿Detección mágica?
Usaremos un árbol sin límites, que memoriza muy bien. Antes de ejecutar: ¿qué versión dará mejor detección en "su" prueba? ¿Cuál dice la verdad?

✍️ *Mi apuesta:*

In [ ]:
# ❌ MAL: sobremuestrear TODO y luego dividir
todos = pd.DataFrame(X).assign(fraude=y)
todos_sobre = pd.concat([todos[todos.fraude == 0],
                         resample(todos[todos.fraude == 1], n_samples=(todos.fraude == 0).sum(), random_state=0)])
Xa, Xb, ya, yb = train_test_split(todos_sobre.drop(columns="fraude"), todos_sobre.fraude, test_size=0.3, random_state=0)
arbol_mal = DecisionTreeClassifier(random_state=0).fit(Xa, ya)
reporte(yb, arbol_mal.predict(Xb), "❌ Re-muestreo antes")

# ✅ BIEN: dividir, sobremuestrear SOLO el entrenamiento y probar con la proporción real
arbol_bien = DecisionTreeClassifier(random_state=0).fit(sobre.drop(columns="fraude"), sobre.fraude)
reporte(y_te, arbol_bien.predict(X_te), "✅ Re-muestreo después")

<details><summary><b>👀 Ver respuesta</b></summary>

- **❌ Antes:** detección **100 %**, precisión ≈98 % y exactitud ≈99 %. Espectacular… y falso. Casi cada fraude de "prueba" tiene copias idénticas en el entrenamiento: el árbol solo tiene que **reconocerlas**.
- **✅ Después:** detección ≈**84 %**, precisión ≈73 %. Es un buen resultado, pero es el **real**: el del árbol frente a fraudes **nuevos**.

Además, en la versión ❌ la prueba tiene **50 % de fraudes**, cuando en la vida real hay un 5 %: la precisión y la exactitud medidas ahí no significan nada.
</details>

---
## 8. 🎛️ Las perillas del desbalance

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `class_weight` | Cuánto "pesa" la clase rara al entrenar | Más detección, menos precisión | Menos detección, más precisión | `"balanced"`, o un diccionario `{0: 1, 1: w}` |
| Umbral de decisión | Probabilidad mínima para decir "clase rara" | Menos alarmas: más precisión, menos detección | Más alarmas: más detección, menos precisión | 0.5 y ajustar según el costo de cada error |
| Proporción de re-muestreo | Cuánto se equilibran las clases | Más equilibrio → más detección (y más riesgo de memorizar copias) | Más cerca de la proporción real | No hace falta llegar a 50/50 |
| `stratify` al dividir | Que cada conjunto tenga la proporción real | — | — | Siempre `stratify=y` |

## 9. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué hacer |
|---|---|---|
| Exactitud altísima, pero el modelo casi nunca predice la clase rara | Desbalance + medir solo exactitud | Mirar detección y precisión; comparar con el perezoso |
| Detección casi perfecta tras re-muestrear | Re-muestreo **antes** de dividir (fuga) | Re-muestrear solo el entrenamiento (o usar pesos) |
| El conjunto de prueba no tiene casos raros | División sin estratificar | `stratify=y` (Lección 2.5) |
| Demasiadas falsas alarmas tras "balancear" | Se exageró el peso o el umbral | Elegir el punto según el costo de cada error |
| Se tiró el 95 % de los datos | Submuestreo con pocos datos | Usar pesos de clase o mover el umbral |

### 🔮 Ejercicio 8 — ¿Cuánto vale un modelo?
Un modelo de abandono de clientes (10 % de abandonos) tiene **exactitud 90 %**. Sin ver nada más, ¿es un buen modelo?

<details><summary><b>👀 Ver respuesta</b></summary>

**No se puede saber, y es sospechoso.** 90 % es justo lo que lograría el modelo perezoso que dice "nadie se va". Antes de opinar, pide la **detección** y la **precisión** de la clase "se va", y compáralas con el perezoso.
</details>

---
## 10. 🧠 Resumen en 5 puntos
1. Con clases desbalanceadas, la **exactitud engaña**: decir siempre "normal" ya da un número alto.
2. Mira los errores por separado: **detección** (¿cuántos raros atrapé?) y **precisión** (¿cuántas de mis alarmas eran reales?).
3. Estrategias: **pesos de clase** (la primera opción), **re-muestreo** (sobre/sub/SMOTE) y **mover el umbral**.
4. Todas negocian el mismo equilibrio: **más detección = más falsas alarmas**. Elige según el **costo** de cada error.
5. Re-muestrea **solo el entrenamiento**, después de dividir (mejor aún, dentro de un pipeline); la prueba conserva la proporción real.

---
## 11. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 11.1 Repaso rápido (opción múltiple)

**1.** En un problema con 3 % de casos positivos, un modelo tiene 97 % de exactitud. ¿Qué puedes concluir?
- a) Es excelente · b) Podría no detectar ningún positivo · c) Tiene 97 % de detección · d) Está sobreajustado

**2.** De 50 enfermos reales, el modelo detecta 40. De sus 100 alarmas, 40 son enfermos. ¿Detección y precisión?
- a) 80 % y 40 % · b) 40 % y 80 % · c) 80 % y 80 % · d) 40 % y 40 %

**3.** Usas `class_weight="balanced"`. ¿Qué esperas?
- a) Más detección y más precisión · b) Más detección y menos precisión · c) Menos detección y más precisión · d) Nada cambia

**4.** ¿Cuándo se debe re-muestrear?
- a) Antes de dividir · b) Solo sobre el entrenamiento, después de dividir · c) Sobre la prueba · d) Sobre todo

**5.** Bajas el umbral de 0.5 a 0.2. ¿Qué pasa con las falsas alarmas?
- a) Bajan · b) Suben · c) No cambian · d) Desaparecen

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** 97 % es lo que da decir siempre "negativo". Necesitas la detección.
2. **a)** Detección 40/50 = 80 %; precisión 40/100 = 40 %.
3. **b)** Se negocia precisión por detección.
4. **b)** Si re-muestreas antes, las copias llegan a la prueba (fuga).
5. **b)** El modelo dispara con menos certeza, así que dispara más veces.
</details>

### 11.2 Preguntas para pensar (sin ejecutar)

**A.** En un filtro de spam, ¿qué es peor: un spam que llega a tu bandeja o un correo importante que se va a la carpeta de spam? ¿Qué te dice eso sobre si priorizar detección o precisión?

**B. Caso de análisis 💼** Una fábrica tiene sensores en 200 máquinas y quiere predecir fallas para el día siguiente. Las fallas son el 0.5 % de los días-máquina. Revisar una máquina cuesta 2,000; una falla no prevista cuesta 150,000. Un analista propone submuestrear los días normales hasta dejar 50/50. ¿Qué opinas? ¿Qué propondrías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Para la mayoría de las personas es peor **perder un correo importante** (una falsa alarma del filtro). Por eso los filtros de spam priorizan la **precisión**: solo mandan a spam si están muy seguros, aunque se les escape algún spam. Es un ejemplo de que **no siempre** hay que maximizar la detección: depende del costo de cada error.

**B.** El submuestreo 50/50 tiraría el **99.5 %** de los días normales: muchísima información sobre cómo se ve una máquina sana. Mejor: **pesos de clase** o **mover el umbral** usando todos los datos, y elegir el umbral con los costos: una falla escapada cuesta 75 veces más que una revisión, así que conviene revisar aunque la probabilidad de falla sea baja (alrededor de 2,000 / 152,000 ≈ 1.3 %). Además: estratificar o dividir **por fecha** (es una serie de tiempo) y **por máquina** si se quieren predecir máquinas nuevas.
</details>

### 11.3 🧩 Práctica integradora: "El costo del fraude"

Un banco te da 20,000 transacciones con **≈2 % de fraudes**. La regla de negocio:

> 💸 **Un fraude que se escapa cuesta 5,000. Revisar una alarma (falsa o no) cuesta 100.**

Tu objetivo: el **menor costo total**, no la mayor exactitud.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Prepara los datos y la función de costo**

In [ ]:
X_b, y_b = make_classification(n_samples=20_000, n_features=10, n_informative=5, n_redundant=2,
                               weights=[0.98, 0.02], class_sep=2.0, flip_y=0.005, random_state=1)
X_b_tr, X_b_te, y_b_tr, y_b_te = train_test_split(X_b, y_b, test_size=0.25, stratify=y_b, random_state=0)

def costo(y_real, y_pred):
    """Costo total: 5,000 por fraude escapado + 100 por cada alarma revisada."""
    _, fp, fn, vp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    return 5_000 * fn + 100 * (vp + fp)

print(f"Fraudes en entrenamiento: {y_b_tr.sum()} de {len(y_b_tr)} ({y_b_tr.mean():.1%})")

**Paso 2 🔮 — Las estrategias "tontas".** Antes de ejecutar, calcula en papel (con los datos de entrenamiento): ¿cuánto cuesta **no revisar nada**? ¿Y **revisar todo**?

✍️ *Mis cálculos:*

In [ ]:
print(f"No revisar nada: {costo(y_b_tr, np.zeros(len(y_b_tr))):>12,}")
print(f"Revisar todo   : {costo(y_b_tr, np.ones(len(y_b_tr))):>12,}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **No revisar nada:** todos los fraudes se escapan: 338 × 5,000 = **1,690,000**.
- **Revisar todo:** 15,000 revisiones × 100 = **1,500,000**.

Las dos cuestan casi lo mismo y son malas. Cualquier modelo que propongas debe costar **mucho menos** que ≈1.5 millones.
</details>

**Paso 3 ▶️ — Normal vs. balanceada (sin tocar la prueba).** Usamos validación cruzada sobre el entrenamiento (`cross_val_predict` da, para cada transacción, la predicción de un modelo que **no la vio**).

In [ ]:
normal = make_pipeline(StandardScaler(), LogisticRegression())
balanc = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced"))

for nombre, modelo in [("Logística normal", normal), ("Logística balanceada", balanc)]:
    pred = cross_val_predict(modelo, X_b_tr, y_b_tr, cv=5)
    reporte(y_b_tr, pred, nombre)
    print(f"{'':<22} costo total: {costo(y_b_tr, pred):,}\n")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Normal:** detección ≈**53 %**, costo ≈**0.82 millones**: se escapa casi la mitad de los fraudes.
- **Balanceada:** detección ≈**85 %**, costo ≈**0.45 millones**: muchas más falsas alarmas (≈1,600), pero cada una cuesta solo 100, mientras que cada fraude escapado cuesta 5,000.

Con estos costos, la balanceada gana por mucho, aunque su exactitud es **mucho peor**.
</details>

**Paso 4 ✏️ — Busca el mejor umbral.** Con las probabilidades de la logística **normal**, prueba umbrales y grafica el costo. ¿Cuál es el mejor?

In [ ]:
prob_cv = cross_val_predict(normal, X_b_tr, y_b_tr, cv=5, method="predict_proba")[:, 1]

umbrales = np.arange(0.01, 0.51, 0.01)                         # ✏️
costos = [costo(y_b_tr, (prob_cv >= u).astype(int)) for u in umbrales]
mejor = umbrales[int(np.argmin(costos))]

plt.plot(umbrales, np.array(costos) / 1e6, "o-", ms=3)
plt.axvline(mejor, color="r", ls="--", label=f"mejor umbral ≈ {mejor:.2f}")
plt.xlabel("umbral"); plt.ylabel("costo total (millones)"); plt.legend(); plt.show()
print(f"Mejor umbral: {mejor:.2f} → costo {min(costos):,}")
reporte(y_b_tr, (prob_cv >= mejor).astype(int), f"umbral {mejor:.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El costo mínimo aparece con un umbral **muy bajo**, ≈**0.02** (costo ≈**0.47 millones**, detección ≈85 %): prácticamente lo mismo que la balanceada (0.45), por otro camino. Tiene lógica: si un fraude escapado cuesta **50 veces** más que una revisión, conviene revisar en cuanto la probabilidad de fraude supere aproximadamente 100 / 5,000 = **2 %**.

La curva tiene forma de "U": con umbrales muy bajos revisas casi todo (muchas revisiones); con umbrales altos se escapan muchos fraudes.
</details>

**Paso 5 ▶️ — El examen final, una sola vez.** Entrenamos con todo el entrenamiento y evaluamos en la prueba, que conserva la proporción **real** de fraudes.

In [ ]:
UMBRAL = mejor                     # ✏️ o el que hayas elegido

final = normal.fit(X_b_tr, y_b_tr)
pred_final = (final.predict_proba(X_b_te)[:, 1] >= UMBRAL).astype(int)
reporte(y_b_te, pred_final, "Final en prueba")
print(f"Costo en prueba     : {costo(y_b_te, pred_final):>10,}")
print(f"No revisar nada     : {costo(y_b_te, np.zeros(len(y_b_te))):>10,}")
print(f"Revisar todo        : {costo(y_b_te, np.ones(len(y_b_te))):>10,}")

<details><summary><b>👀 Ver respuesta</b></summary>

En prueba, el modelo cuesta ≈**0.17 millones**, contra 0.56 de no revisar nada y 0.50 de revisar todo: **le ahorra al banco cerca del 70 %** del costo. Detecta ≈82 % de los fraudes con una precisión baja (≈13 %), y está bien: así lo pide la regla de negocio.

**La idea central del módulo en un ejercicio:** la mejor decisión no salió de la exactitud, sino de entender **el costo de cada error** (Lección 4.8).
</details>

---
## 📚 Para profundizar
- 📘 **Documentación:** `imbalanced-learn` (SMOTE y otras técnicas), guía de usuario.
- 📘 **Documentación:** scikit-learn, *TunedThresholdClassifierCV* (elegir el umbral automáticamente con validación cruzada).
- 📺 **Video:** StatQuest, "Sensitivity and Specificity" y "ROC and AUC" (inglés, con subtítulos).

---
⬅️ **Anterior:** [2.6 Fuga de datos (*data leakage*): el error silencioso más caro](2.6-fuga-de-datos.ipynb) · ➡️ **Siguiente:** [2.8 La maldición de la dimensionalidad (intuición)](2.8-maldicion-de-la-dimensionalidad.ipynb)